# Neural Criticality and Avalanche Analysis

Detecting and analyzing critical dynamics in neural systems.

**Topics:**
- Neuronal avalanche detection
- Power law analysis (size and duration distributions)
- Branching parameter estimation (σ ≈ 1 at criticality)
- Self-organized criticality tests
- Distance from criticality (DCC)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import linregress, kstest
from neuros_mechint.fractals import (
    NeuronalAvalanche,
    BranchingProcess,
    CriticalityDetector,
    SelfOrganizedCriticality
)
from neuros_mechint.visualization import CriticalityVisualizer

%matplotlib inline

## 1. Neuronal Avalanche Detection

Identify spatiotemporal cascades of neural activity.

In [ ]:
# Generate synthetic neural activity at criticality
# (Branching process with σ = 1)
n_neurons = 100
n_timesteps = 10000
branching_param = 1.0  # Critical

# Simulate branching process
activity = np.zeros((n_timesteps, n_neurons))
activity[0, 0] = 1  # Initial seed

for t in range(1, n_timesteps):
    active_neurons = np.where(activity[t-1] > 0)[0]
    for neuron in active_neurons:
        # Each active neuron activates ~branching_param neighbors
        n_offspring = np.random.poisson(branching_param)
        targets = np.random.choice(n_neurons, size=min(n_offspring, n_neurons), replace=False)
        activity[t, targets] = 1

print(f"Generated {n_timesteps} timesteps for {n_neurons} neurons")
print(f"Mean activity: {activity.mean():.4f}")
print(f"Total spikes: {activity.sum():.0f}")

# Detect avalanches
detector = NeuronalAvalanche(threshold=0.0)  # Any activity counts
avalanches = detector.detect_avalanches(activity)

print(f"\nDetected {len(avalanches)} avalanches")

# Visualize
viz = CriticalityVisualizer()
fig = viz.plot_avalanche_raster(
    activity=activity[:1000],  # First 1000 timesteps
    avalanches=avalanches[:20],  # First 20 avalanches
    title="Neuronal Avalanches (Critical Dynamics)"
)
fig.show()

## 2. Power Law Analysis

At criticality, avalanche sizes follow power law: P(s) ∝ s^(-τ)

In [ ]:
# Extract avalanche statistics
sizes = np.array([av['size'].sum() for av in avalanches if av['size'].sum() > 0])
durations = np.array([av['duration'] for av in avalanches if av['duration'] > 0])

print(f"Avalanche statistics:")
print(f"  Mean size: {sizes.mean():.2f} spikes")
print(f"  Max size: {sizes.max():.0f} spikes")
print(f"  Mean duration: {durations.mean():.2f} timesteps")
print(f"  Max duration: {durations.max():.0f} timesteps")

# Fit power law to sizes
# Use logarithmic binning
log_sizes = np.log10(sizes[sizes > 0])
counts, bins = np.histogram(log_sizes, bins=20)
counts = counts[counts > 0]
bin_centers = (bins[:-1] + bins[1:])[counts > 0] / 2
log_counts = np.log10(counts)

# Linear fit in log-log space
slope, intercept, r_value, p_value, std_err = linregress(bin_centers, log_counts)
tau = -slope  # Power law exponent

print(f"\nPower law fit:")
print(f"  Exponent τ = {tau:.3f} (theory predicts ~1.5 for criticality)")
print(f"  R² = {r_value**2:.4f}")
print(f"  p-value = {p_value:.4e}")

# Visualize power law
fig = viz.plot_power_law_distribution(
    sizes=sizes,
    title="Avalanche Size Distribution",
    fit_params={'tau': tau, 'r_squared': r_value**2}
)
fig.show()

# Size-duration relationship
# At criticality: S ∝ D^γ with γ ≈ 2
valid = (sizes > 0) & (durations > 0)
log_S = np.log10(sizes[valid])
log_D = np.log10(durations[valid])

slope_SD, _, r_SD, _, _ = linregress(log_D, log_S)
gamma = slope_SD

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.loglog(durations[valid], sizes[valid], 'o', alpha=0.5, markersize=4)
D_range = np.logspace(np.log10(durations[valid].min()), np.log10(durations[valid].max()), 100)
S_fit = 10**(intercept) * D_range**gamma
plt.loglog(D_range, S_fit, 'r--', linewidth=2, 
           label=f'γ = {gamma:.2f}, R² = {r_SD**2:.3f}')
plt.xlabel('Duration (timesteps)')
plt.ylabel('Size (spikes)')
plt.title('Size-Duration Relationship')
plt.legend()
plt.grid(True, alpha=0.3)

# Duration distribution
plt.subplot(1, 2, 2)
unique_durations, duration_counts = np.unique(durations, return_counts=True)
plt.loglog(unique_durations, duration_counts, 'o-', markersize=6)
plt.xlabel('Duration (timesteps)')
plt.ylabel('Count')
plt.title('Duration Distribution')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSize-duration scaling: γ = {gamma:.3f} (theory predicts ~2.0)")

## 3. Branching Parameter Estimation

At criticality, σ = 1 (each active neuron activates exactly 1 neuron on average).

In [ ]:
# Estimate branching parameter
bp = BranchingProcess()
sigma = bp.estimate_branching_parameter(activity)

print(f"Estimated branching parameter σ = {sigma:.4f}")
print(f"Distance from criticality: |σ - 1| = {abs(sigma - 1):.4f}")
print(f"State: ", end="")
if abs(sigma - 1) < 0.05:
    print("Critical ✓")
elif sigma < 1:
    print("Subcritical (activity dies out)")
else:
    print("Supercritical (activity explodes)")

# Test different σ values
test_sigmas = [0.8, 0.9, 1.0, 1.1, 1.2]
avalanche_stats = []

for test_sigma in test_sigmas:
    # Simulate
    test_activity = np.zeros((1000, n_neurons))
    test_activity[0, 0] = 1
    
    for t in range(1, 1000):
        active = np.where(test_activity[t-1] > 0)[0]
        for neuron in active:
            n_offspring = np.random.poisson(test_sigma)
            targets = np.random.choice(n_neurons, size=min(n_offspring, n_neurons), replace=False)
            test_activity[t, targets] = 1
    
    # Analyze
    test_avalanches = detector.detect_avalanches(test_activity)
    test_sizes = [av['size'].sum() for av in test_avalanches if av['size'].sum() > 0]
    
    avalanche_stats.append({
        'sigma': test_sigma,
        'n_avalanches': len(test_avalanches),
        'mean_size': np.mean(test_sizes) if test_sizes else 0,
        'max_size': np.max(test_sizes) if test_sizes else 0
    })

# Plot
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot([s['sigma'] for s in avalanche_stats],
         [s['n_avalanches'] for s in avalanche_stats], 'o-', linewidth=2, markersize=8)
plt.axvline(1.0, color='r', linestyle='--', label='Critical')
plt.xlabel('Branching Parameter σ')
plt.ylabel('Number of Avalanches')
plt.title('Avalanche Count vs σ')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot([s['sigma'] for s in avalanche_stats],
         [s['mean_size'] for s in avalanche_stats], 'o-', linewidth=2, markersize=8)
plt.axvline(1.0, color='r', linestyle='--', label='Critical')
plt.xlabel('Branching Parameter σ')
plt.ylabel('Mean Avalanche Size')
plt.title('Mean Size vs σ')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot([s['sigma'] for s in avalanche_stats],
         [s['max_size'] for s in avalanche_stats], 'o-', linewidth=2, markersize=8)
plt.axvline(1.0, color='r', linestyle='--', label='Critical')
plt.xlabel('Branching Parameter σ')
plt.ylabel('Max Avalanche Size')
plt.title('Max Size vs σ')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Criticality Detection in Foundation Models

Test if neural network activations exhibit critical dynamics.

In [ ]:
# Simulate foundation model activations
# (In practice, extract from real model)
model_activity = torch.randn(1000, 256)  # 1000 timesteps, 256 units

# Threshold to create binary "spikes"
threshold = model_activity.mean() + model_activity.std()
binary_activity = (model_activity > threshold).float().numpy()

print(f"Model activity statistics:")
print(f"  Spike rate: {binary_activity.mean():.4f}")
print(f"  Total spikes: {binary_activity.sum():.0f}")

# Run criticality detector
crit_detector = CriticalityDetector()
results = crit_detector.test_criticality(binary_activity)

print(f"\nCriticality Test Results:")
print(f"  Branching parameter: {results['branching_parameter']:.4f}")
print(f"  Power law exponent: {results.get('power_law_exponent', 'N/A')}")
print(f"  Is critical: {results['is_critical']}")
print(f"  DCC (distance from criticality): {results['DCC']:.4f}")

# Visualize
model_avalanches = detector.detect_avalanches(binary_activity)
model_sizes = [av['size'].sum() for av in model_avalanches if av['size'].sum() > 0]

fig = viz.plot_power_law_distribution(
    sizes=np.array(model_sizes),
    title="Foundation Model Avalanche Distribution"
)
fig.show()

# Compare to critical simulation
print(f"\nComparison to Critical System:")
print(f"  Model σ: {results['branching_parameter']:.4f}")
print(f"  Critical σ: ~1.000")
print(f"  Interpretation: Model is {'near' if abs(results['branching_parameter'] - 1) < 0.1 else 'far from'} criticality")

## Conclusion

This notebook demonstrated:
1. **Avalanche detection**: Identifying spatiotemporal cascades
2. **Power law analysis**: Testing for scale-free dynamics
3. **Branching parameter**: Quantifying distance from criticality
4. **Foundation model testing**: Applying criticality analysis to AI

Critical dynamics optimize information processing, dynamic range, and computational capacity. Testing for criticality in foundation models reveals whether they exhibit brain-like self-organization.